<img src='./img/up_bann.png' style="display: block; margin: 0 auto;">

*This algorithm has been **community** contributed to APEx and has not undergine scientific validation review* 

# Ship detection — Dnipro-Bug estuary, Mykolaiv (September 2023)

This notebook runs the `ship_detection` algorithm over the port of Mykolaiv and the Southern Bug river. It finds water using **Sentinel-2** and flags bright **Sentinel-1** radar returns on that water as vessels.

The `ship_detection` UDP (Sentinel-2 NDWI water mask combined with bright Sentinel-1 VV/VH backscatter over water) on CDSE over the port of Mykolaiv and the Southern Bug river, then plots the result.

Scenes picked up by the window (openEO end dates are exclusive; the UDP keeps the first scene per sensor):

| Window | Sentinel-1 GRD | Sentinel-2 L1C |
|---|---|---|
| 2023-09-03 → 2023-09-07 | 2023-09-05 (descending, full bbox) | 2023-09-04 (0–4% cloud) |

### Import Required libraries

In [ ]:
from pathlib import Path
import os
from getpass import getpass
from esa_apex_toolbox.algorithms import GithubAlgorithmRepository

import openeo

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

### Define Costum Functions

In [ ]:
def read_rgb(path):
    """Read a 3-band ship_detection GeoTIFF as an (H, W, 3) 0-1 array plus its extent in km."""
    with rasterio.open(path) as src:
        data = src.read()
        b = src.bounds
    # The UDP output is already display-scaled; only clip to 0-1.
    rgb = np.clip(np.nan_to_num(np.moveaxis(data, 0, -1)), 0, 1)
    extent = [b.left / 1000, b.right / 1000, b.bottom / 1000, b.top / 1000]
    return rgb, extent, data

### Authentication

To run the jobs directly in the openEO CDSE federation, ir is required to authenticate with our CDSE credentials:

In [ ]:
connection = openeo.connect("https://openeo.dataspace.copernicus.eu/").authenticate_oidc()

Also, to explore the APEx Catalog, a GitHub token is recommended (GitHub normally allows 60 unauthenticated **REST API requests per hour per IP address**):

In [ ]:
github_token = os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN')
if not github_token:
    github_token = getpass('GitHub token (optional; Enter for unauthenticated access): ') or None

We define our area of interest, the time frame of interest and our output paths

In [ ]:
# Bounding box and temporal extent
shape = {"west": 31.745689, "south": 46.600381, "east": 32.126161, "north": 47.020555}
start_date, end_date = ["2023-09-03", "2023-09-07"]
output = Path("ship_detection_mykolaiv.tiff")

The next step is selecting our algorithm and accessing the `udp_link` trough its `id`. For this case it corresponds to `ship_detection`

In [ ]:
repo = GithubAlgorithmRepository(
    owner='ESA-APEx',
    repo='apex_algorithms',
    folder='algorithm_catalog',
    branch='main',
    token=github_token,
)

process_id = "ship_detection"
sd_algorithm = repo.get_algorithm(process_id)

sd_algorithm

We define our `datacube_from_process` to host the results over the area of interest and time frame we defined.

In [ ]:
ships_mykolaiv = connection.datacube_from_process(
    process_id,
    namespace=sd_algorithm.udp_link.href,
    time=[start_date, end_date],
    bounding_box=shape,
)

## Ship detection

With everything in place, we define our job characteristics, start our job and monitor.

In [ ]:
job = ships_mykolaiv.create_job(
    title=f"Ship detection for {start_date} to {end_date}",
    out_format="GTiff",
)
job.start_and_wait()

Once successfuly executed we can download our results

In [ ]:
# The result is a single GeoTIFF (~3 MB). If the download breaks, rerun only
# this cell; the job does not need to be resubmitted.
job.get_results().download_file(output)

## Detected ships

Based on our defined `read_rgb` funciton, we can explore our results and point at the detected ships!

In [ ]:
rgb, extent, data = read_rgb(output)

# The UDP paints ship pixels pure white (1, 1, 1). Ring them so the 50 m pixels
# stay visible at full-scene scale.
rows, cols = np.nonzero(np.all(data == 1.0, axis=0))
height, width = rgb.shape[:2]
x = extent[0] + (cols + 0.5) / width * (extent[1] - extent[0])
y = extent[3] - (rows + 0.5) / height * (extent[3] - extent[2])

fig, ax = plt.subplots(figsize=(9, 13))
ax.imshow(rgb, extent=extent)
ax.scatter(x, y, s=60, facecolors="none", edgecolors="#e8202a", linewidths=1)
ax.set_title(f"S1 2023-09-05, S2 2023-09-04 — {len(rows)} ship pixels", fontsize=13)
ax.set_xlabel("Easting (km, UTM 36N)")
ax.set_ylabel("Northing (km, UTM 36N)")

fig.legend(
    handles=[
        Line2D([], [], marker="o", linestyle="none", markerfacecolor="none",
               markeredgecolor="#e8202a", label="Ship (S1 VV or VH > 0.3 over water)"),
        Patch(color="#1e4c8c", label="Water (Sentinel-2 NDWI > 0.1)"),
        Patch(color="#9a8a6a", label="Land (Sentinel-2 true colour)"),
    ],
    loc="lower center",
    ncol=1,
    frameon=False,
    fontsize=11,
)
fig.suptitle("Ship detection — Dnipro-Bug estuary, Mykolaiv (September 2023)", fontsize=16)
fig.tight_layout(rect=(0, 0.07, 1, 0.97))
fig.savefig("ship_detection_mykolaiv.png", dpi=200)
plt.show()

The true-colour map of the estuary with each detection circled in red (293 ship pixels on 4–5 September 2023).<br>

It is a base for vessel monitoring: port activity, sanctions or grain-corridor checks, or spotting ships that switch off their AIS transponder. The detector is simple, so it also flags bright objects that aren't ships, such as harbour structures and some spots over the city and small inland ponds. Treat the circles as candidates to check, not confirmed vessels.

<img src='./img/low_bann.png' style="display: block; margin: 0 auto;">